# FightWatch — pilot phát hiện đánh nhau

Chạy trên **GPU Colab**. Notebook tự đồng bộ code, dùng UBI-Fights đã có trên Drive và giới hạn pilot **30 phút** (không gồm cài thư viện/chuẩn bị dữ liệu). Không train toàn dataset, không dùng test chính thức để chọn phương pháp. Checkpoint xuất ra chỉ là **pilot**, chưa được chứng nhận cho demo.

Các lần chạy cũ nằm riêng và được giữ nguyên. Khi cần Google đăng nhập hoặc cấp quyền Drive, chủ tài khoản tự xác nhận.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/SENTINEL")
REPO_URL = "https://github.com/TranDuon/anomaly-behavior-detector-system.git"
DRIVE_REPO = DRIVE_ROOT / "repo" / "anomaly-behavior-detector-system"
LOCAL_REPO = Path("/content/fightwatch-code")
ARCHITECTURE = "x3d_s"
RUN_NAME = "fight_pilot_x3d_v1"
BUDGET_MINUTES = 30
PILOT_VIDEOS = 48
CLIPS_PER_VIDEO = 8
SEED = 42
OUTPUT = DRIVE_ROOT / "artifacts" / RUN_NAME
CACHE = DRIVE_ROOT / "features" / "fight_pilot_v1" / ARCHITECTURE
DATA_ROOT = Path("/content/fightwatch-data/UBI_FIGHTS")
for path in (OUTPUT, CACHE, DRIVE_REPO.parent, DATA_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print("Kết quả:", OUTPUT)
print("Budget pilot:", BUDGET_MINUTES, "phút; video:", PILOT_VIDEOS)


## 1. Đồng bộ code mới nhất

Bản repo trên Drive là bản sao chạy: cell này ghi đè thay đổi code chưa commit của bản sao đó bằng `origin/main`. Dataset và artifact nằm ngoài repo, không bị thay đổi. Code được chép sang `/content` để đọc nhanh hơn.

In [ ]:
import os, shutil, subprocess, sys, json, time

def checked(arguments, **kwargs):
    return subprocess.run([str(value) for value in arguments], check=True, **kwargs)

if (DRIVE_REPO / ".git").exists():
    remote = subprocess.check_output(["git", "-C", str(DRIVE_REPO), "remote", "get-url", "origin"], text=True).strip()
    if remote.removesuffix(".git") != REPO_URL.removesuffix(".git"):
        raise RuntimeError("Remote của repo Drive không đúng. Dừng để tránh ghi nhầm repo.")
    checked(["git", "-C", DRIVE_REPO, "fetch", "origin", "main"])
    checked(["git", "-C", DRIVE_REPO, "reset", "--hard", "origin/main"])
else:
    checked(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, DRIVE_REPO])
shutil.copytree(DRIVE_REPO, LOCAL_REPO, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", "models", "uploads", "logs", ".env", ".pytest_cache"))
os.chdir(LOCAL_REPO)
if str(LOCAL_REPO) not in sys.path:
    sys.path.insert(0, str(LOCAL_REPO))
revision = subprocess.check_output(["git", "-C", str(DRIVE_REPO), "rev-parse", "HEAD"], text=True).strip()
(OUTPUT / "code_revision.txt").write_text(revision)
print("Code revision:", revision)


## 2. Cài thư viện, xác nhận GPU và chạy test logic

Không tự chuyển sang CPU để tránh vô tình chạy một phiên rất lâu.

In [ ]:
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
import torch
assert torch.cuda.is_available(), "Hãy chọn GPU trong Runtime > Change runtime type trước khi tiếp tục."
print("GPU:", torch.cuda.get_device_name(0))
checked([sys.executable, "-m", "pytest", "tests/test_fight_runtime.py", "-q", "-p", "no:cacheprovider"])


## 3. Chuẩn bị một tập nhỏ từ archive đã có

Chỉ giải nén video được chọn, không giải nén và trích đặc trưng toàn bộ ~80 giờ. Video test chính thức bị loại ngay khi chọn. File gốc trên Drive không bị sửa.

In [ ]:
import zipfile
import numpy as np
from anomaly.fight_data import parse_ubi_annotation

archive_path = DRIVE_ROOT / "datasets" / "fight" / "UBI_FIGHTS.zip"
assert archive_path.exists(), f"Thiếu dataset gốc: {archive_path}"
assert archive_path.stat().st_size == 8_117_112_419, "Kích thước archive UBI-Fights không khớp; cần kiểm tra file."
rng = np.random.default_rng(SEED)
exclude_path = DRIVE_ROOT / "artifacts" / "fight" / "dedup" / "exclude.json"
excluded = set(json.loads(exclude_path.read_text())["exclude_from_training"]) if exclude_path.exists() else set()
with zipfile.ZipFile(archive_path) as archive:
    names = archive.namelist()
    test_name = next(name for name in names if name.endswith("/test_videos.csv") or name == "test_videos.csv")
    official_test = {line.strip() for line in archive.read(test_name).decode().splitlines() if line.strip()}
    videos = {Path(name).stem: name for name in names if Path(name).suffix.lower() in {".mp4", ".avi"} and "/videos/" in "/" + name}
    annotations = {Path(name).stem: name for name in names if "/annotation/" in "/" + name and name.endswith(".csv")}
    pools = {False: [], True: []}
    for key in sorted(set(videos) & set(annotations)):
        if key in official_test or "ubi/" + key in excluded:
            continue
        labels = parse_ubi_annotation(archive.read(annotations[key]).decode())
        pools[bool(labels.any())].append(key)
    selected = []
    for positive in (False, True):
        rng.shuffle(pools[positive])
        selected.extend(pools[positive][:PILOT_VIDEOS // 2])
    for index, key in enumerate(selected, 1):
        for member in (videos[key], annotations[key]):
            parts = Path(member).parts
            marker = "videos" if "videos" in parts else "annotation"
            destination = DATA_ROOT.joinpath(*parts[parts.index(marker):])
            destination.parent.mkdir(parents=True, exist_ok=True)
            expected = archive.getinfo(member).file_size
            if not destination.exists() or destination.stat().st_size != expected:
                temporary = destination.with_suffix(destination.suffix + ".partial")
                with archive.open(member) as source, temporary.open("wb") as target:
                    shutil.copyfileobj(source, target)
                temporary.replace(destination)
        if index % 8 == 0:
            print(f"Đã chuẩn bị {index}/{len(selected)} video", flush=True)
    (DATA_ROOT / "test_videos.csv").write_bytes(archive.read(test_name))
(OUTPUT / "selected_videos.json").write_text(json.dumps(selected, indent=2))
print("Số video pilot:", len(selected), "| Không dùng official test.")


## 4. Pilot có ngân sách và log trực tiếp

Các bước: kiểm tra input/crop → thử học tập nhỏ → train đầu phân loại → fine-tune ngắn → đánh giá validation. Nếu hết ngân sách, kết luận **inconclusive**, không tự chạy train lớn.

Cache clip và feature được dùng lại khi chạy lại cùng cấu hình. Không coi một lần pilot đạt là bằng chứng chất lượng cuối cùng.

In [ ]:
import threading
command = [sys.executable, "-u", "-m", "scripts.fight_pilot",
    "--dataset-root", str(DATA_ROOT), "--output-dir", str(OUTPUT), "--cache-dir", str(CACHE),
    "--architecture", ARCHITECTURE, "--budget-minutes", str(BUDGET_MINUTES),
    "--videos", str(PILOT_VIDEOS), "--clips-per-video", str(CLIPS_PER_VIDEO),
    "--seed", str(SEED), "--device", "cuda", "--exclude", str(exclude_path)]
start = time.monotonic()
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
def stop_over_budget():
    if process.poll() is None:
        process.terminate()
        threading.Timer(10, lambda: process.kill() if process.poll() is None else None).start()
watchdog = threading.Timer(BUDGET_MINUTES * 60 + 120, stop_over_budget)
watchdog.start()
try:
    with (OUTPUT / "run_log.txt").open("a", encoding="utf-8") as log:
        log.write(f"\\nREVISION {revision} | {time.ctime()}\\n")
        for line in process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
    return_code = process.wait()
finally:
    watchdog.cancel()
if return_code != 0:
    failure = {"decision": "inconclusive", "reason": "Process failed or exceeded wall-clock limit; inspect run_log.txt.",
               "return_code": return_code, "seconds": time.monotonic() - start, "approved_for_demo": False}
    (OUTPUT / "execution_failure.json").write_text(json.dumps(failure, indent=2))
    raise RuntimeError(f"Pilot chưa hoàn tất (exit {return_code}). Log: {OUTPUT / 'run_log.txt'}")


## 5. Xem kết luận, ảnh crop và báo cáo

Kiểm tra crop có đúng người, cùng video có cả nhãn đánh nhau và bình thường. `promising` chỉ cho phép cân nhắc train lớn; chưa có đánh giá test độc lập nên `approved_for_demo` luôn là `False`.

In [ ]:
from IPython.display import display, Image
report = json.loads((OUTPUT / "report.json").read_text())
print(json.dumps(report, ensure_ascii=False, indent=2))
if (OUTPUT / "validation.png").exists():
    display(Image(filename=str(OUTPUT / "validation.png")))
for preview in sorted((OUTPUT / "previews").glob("*.jpg"))[:8]:
    print(preview.name)
    display(Image(filename=str(preview)))
print("Kết quả và log đã lưu trên Drive:", OUTPUT)


## 6. Tải gói thử về local

`run_local.cmd` tự tìm ZIP này trong Downloads. Trong giao diện, phải chủ động bật **Cho phép checkpoint pilot** để thử. Không tự thay model đã nghiệm thu bằng một model chưa đạt.

In [ ]:
from google.colab import files
bundle = OUTPUT / "sentinel_fight_only_runtime.zip"
if bundle.exists() and report.get("threshold") is not None:
    files.download(str(bundle))
else:
    print("Chưa có runtime bundle vì pilot chưa hoàn thành một epoch validation. Xem report/run_log trước khi chạy tiếp.")
